# Домашнее задание блока 2. Лекции 3–4

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IlyaChichkanov/Reinforcement-learning/blob/main/homeworks/hw2/homework.ipynb)

Задание по лекциям 3 и 4: обучение без модели — Q-learning и SARSA, их цели обновления, on-policy и off-policy, Expected SARSA. Среды меняются от части к части, а алгоритмы — те же, что вы написали на семинаре 4: **возьмите `epsilon_greedy`, `sarsa_step`, `q_step` и `train` из своего решения семинара**, это и задумано.

Всего **100 баллов**:

| Часть | Среда | Что проверяем | Баллы |
|---|---|---|---|
| 1. Q-learning на озере 8×8 | знакомая карта из домашки 1 | без модели ценность известна только там, где агент был | 25 |
| 2. Такси по чужому логу | Taxi-v4, лог случайной стратегии | off-policy учится на чужих данных, on-policy — нет | 25 |
| 3. Expected SARSA на такси | Taxi-v4 | среднее по стратегии вместо жребия чинит SARSA | 20 |
| 4. Горная машина на сетке | MountainCar-v0 | таблица на непрерывном состоянии: когда сетка работает, а когда нет | 20 |
| 5. Теория | — | почему всё это работает | 10 |

Части с `assert` проверяются автоматически при запуске ячейки: если `assert` не упал, часть засчитана. Графики и письменные ответы проверяются вручную — пишите их прямо в markdown-ячейках вместо слова `TODO`.

Целиком ноутбук считается около трёх минут (дольше всего — горная машина в части 4). Задание рассчитано на 2–3 часа.

**Не забудьте про квизы.** К этому блоку их два — по лекции 3 и по лекции 4. Пройдите оба в зачётном режиме и положите коды результатов в `quiz/week03.txt` и `quiz/week04.txt`.

**Дедлайн: 3 ноября 2026, 23:59 МСК.**

Первые две ячейки служебные: установка пакетов в Colab и подключение модулей с картинками. Запустите и идите дальше.

In [ ]:
# Если ноутбук открыт в Google Colab: ставим недостающие пакеты. Локально (после uv sync) ячейка ничего не делает.
import importlib.util, subprocess, sys
if importlib.util.find_spec("gymnasium") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium[toy-text,classic-control]"], check=True)

In [ ]:
# Служебное: три модуля с картинками и обвязкой. Читать не нужно, сигнатуры — в docstring каждого.
import pathlib, sys, urllib.request

MODULES = {"lake_plots": "01-intro/seminar", "taxi_plots": "04-sarsa/seminar", "hw2_plots": "homeworks/hw2"}
for name, folder in MODULES.items():
    local = next((p for p in ["", f"../../{folder}", f"../../../{folder}", folder] if (pathlib.Path(p) / f"{name}.py").exists()), None)
    if local is None:                                              # в Colab скачиваем из репозитория курса
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/IlyaChichkanov/Reinforcement-learning/main/{folder}/{name}.py", f"{name}.py")
        local = ""
    sys.path.insert(0, str(pathlib.Path(local).resolve()))

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from lake_plots import lake_desc, draw_lake, draw_policy, plot_values
from taxi_plots import make_taxi, draw_taxi, play_episode, average_greedy_return
from hw2_plots import (mdp_from_gym, success_rate_greedy, greedy_path, collect_random_log, plot_learning_curves,
                       make_mountain_car, greedy_episode, plot_value_grid, plot_positions)

## Часть 1. Q-learning на озере 8×8 (25 баллов)

*Это раздел 2 семинара 4 на карте из домашки 1.* В домашке 1 Cross-Entropy выучил путь по озеру 8×8 за $30 \times 300 = 9000$ эпизодов, а value iteration нашёл точную $Q^*$ по модели. Теперь между ними встанет Q-learning: модели у него нет, зато, в отличие от Cross-Entropy, он учится на каждом шаге.

### 1.1 Инструменты семинара (10 баллов)

Перенесите из своего решения семинара 4 четыре функции: `epsilon_greedy`, `sarsa_step`, `q_step` и `train`. Ниже под ними те же проверки, что были на семинаре, плюс одна новая: `train` обучает обе цели на обрыве.

In [ ]:
GAMMA = 0.95

def epsilon_greedy(Q, s, eps, rng):
    # TODO 1.1: из семинара 4 — с вероятностью eps случайное действие, иначе лучшее по строке Q[s] (ничьи — случайно)
    raise NotImplementedError

def sarsa_step(Q, s, a, r, s2, a2, done, alpha, gamma):
    # TODO 1.1: цель r + gamma * Q[s2, a2] (при done — просто r), затем сдвиг Q[s, a] к цели на долю alpha
    raise NotImplementedError

def q_step(Q, s, a, r, s2, a2, done, alpha, gamma):
    # TODO 1.1: то же, но в цели максимум строки Q[s2]; a2 не используется
    raise NotImplementedError

def train(env, step_fn, n_episodes=500, eps=0.1, alpha=0.5, gamma=1.0, seed=0,
          max_steps=500, eps_decay=1.0, eps_min=0.0):
    # TODO 1.1: цикл обучения из семинара 4 — возвращает (Q, массив return'ов по эпизодам)
    raise NotImplementedError

In [ ]:
# Самопроверка 1.1 — числа из лекции 4
rng = np.random.default_rng(0)
Q_demo = np.array([[0.10, -1.00, 0.20, 0.40]])
assert all(epsilon_greedy(Q_demo, 0, 0.0, rng) == 3 for _ in range(100)), "при eps = 0 выбор всегда жадный"
freq = np.mean([epsilon_greedy(Q_demo, 0, 0.2, rng) == 3 for _ in range(100_000)])
assert 0.84 < freq < 0.86, f"при eps = 0.2 лучшее действие выбирается с частотой 0.85, у вас {freq:.3f}"

Q = np.zeros((2, 4)); Q[0, 0] = 0.30; Q[1, 2] = 0.50; Q[1, 3] = 0.90
sarsa_step(Q, s=0, a=0, r=0.0, s2=1, a2=2, done=False, alpha=0.1, gamma=1.0)
assert np.isclose(Q[0, 0], 0.32), f"SARSA: 0.30 + 0.1 · (0.50 − 0.30) = 0.32, у вас {Q[0, 0]:.3f}"
Q[0, 0] = 0.30
q_step(Q, s=0, a=0, r=0.0, s2=1, a2=2, done=False, alpha=0.1, gamma=1.0)
assert np.isclose(Q[0, 0], 0.36), f"Q-learning: 0.30 + 0.1 · (0.90 − 0.30) = 0.36, у вас {Q[0, 0]:.3f}"

cliff = gym.make("CliffWalking-v1")
Q_sarsa, ret_sarsa = train(cliff, sarsa_step, seed=2)
Q_ql, ret_ql = train(cliff, q_step, seed=2)
assert Q_ql.shape == (48, 4) and len(ret_ql) == 500, "train возвращает таблицу 48 × 4 и return каждого эпизода"
assert ret_sarsa[-100:].mean() > ret_ql[-100:].mean(), \
    "на обрыве при обучении SARSA набирает больше Q-learning — он учитывает свои случайные шаги"
print(f"OK: обрыв, return последних 100 эпизодов {ret_ql[-100:].mean():+.1f} (Q-learning) "
      f"против {ret_sarsa[-100:].mean():+.1f} (SARSA)")

### 1.2 Q-learning против точного ответа (15 баллов)

Озеро 8×8 без скольжения — то же, что в домашке 1. Готовая `mdp_from_gym` достаёт из среды модель $P[s, a, s']$, $R[s, a]$ в формате лекции 2, так что **точную** $Q^*$ можно посчитать value iteration из домашки 1 — перенесите оттуда `bellman_update` и `value_iteration`. Терминальные клетки в модели ведут сами в себя с нулевой наградой, менять ничего не нужно.

Затем обучите Q-learning вашей функцией `train`: $\varepsilon = 0.1$, $\alpha = 0.5$, $\gamma = 0.95$, не больше 100 шагов в эпизоде. Сделайте это для 500, 1000, 2000 и 5000 эпизодов и для каждой таблицы напечатайте долю успехов **жадной** стратегии (`success_rate_greedy`) и номер первого успешного эпизода при обучении (первый эпизод с положительным return).

Для таблицы после 5000 эпизодов нарисуйте в ряд три картинки: точную $V^* = \max_a Q^*$, выученную $V = \max_a Q$ (обе через `plot_values`, с одним и тем же `vmax`) и жадную стратегию (`draw_policy`). Под рисунком напечатайте среднюю ошибку $|V - V^*|$ двумя способами: по всем нетерминальным клеткам и только по клеткам жадного пути (`greedy_path`).

Ответьте текстом: (1) где выученная $V$ совпала с точной, а где осталась нулевой — и почему; (2) сколько эпизодов понадобилось Q-learning до стопроцентной жадной стратегии и как это соотносится с 9000 эпизодами Cross-Entropy из домашки 1 — за счёт чего разница.

In [ ]:
lake = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False)
desc = lake_desc(lake)
P, R, terminal = mdp_from_gym(lake)
cells = np.flatnonzero(~terminal)                     # нетерминальные клетки — по ним считаем ошибки

def bellman_update(Q, P, R, gamma=GAMMA):
    # TODO 1.2: как в домашке 1 (семинар 2)
    raise NotImplementedError

def value_iteration(P, R, gamma=GAMMA, tol=1e-10, max_iter=1000):
    # TODO 1.2: как в домашке 1; вернуть Q* и число итераций
    raise NotImplementedError

Q_star, n_iter = value_iteration(P, R)
V_star = Q_star.max(axis=1)

# TODO 1.2: для n_episodes в (500, 1000, 2000, 5000) обучить Q-learning (train с q_step, eps=0.1, alpha=0.5,
#           gamma=GAMMA, max_steps=100, seed=0), напечатать долю успехов жадной стратегии и первый успешный эпизод;
#           таблицу после 5000 эпизодов сохранить в Q_lake
Q_lake = None
# TODO 1.2: три картинки в ряд (V*, V выученная, жадная стратегия) и две ошибки под ними

In [ ]:
# Самопроверка 1.2
assert Q_star.shape == (64, 4) and np.isclose(V_star[0], 0.95 ** 13), "V*(старт) = 0.95^13: кратчайший путь — 14 шагов"
assert np.allclose(Q_star, bellman_update(Q_star, P, R), atol=1e-6), "Q* должна удовлетворять уравнению оптимальности"
assert Q_lake is not None and Q_lake.shape == (64, 4), "Q_lake — таблица 64 × 4 после 5000 эпизодов"
assert success_rate_greedy(lake, Q_lake) == 1.0, "жадная стратегия после 5000 эпизодов должна доходить всегда"
_path = greedy_path(lake, Q_lake)
assert np.abs(Q_lake.max(axis=1) - V_star)[_path].mean() < 0.01, \
    "вдоль жадного пути выученная V должна совпасть с V*: среда без случайности, α = 0.5 — ошибки нет"
print(f"OK: жадный путь из {len(_path) - 1} шагов, V(старт) = {Q_lake.max(axis=1)[0]:.4f} при точном {V_star[0]:.4f}")

_Ваш ответ:_ TODO

## Часть 2. Такси по чужому логу (25 баллов)

*Это раздел 5 лекции 4 на такси из семинара 4.* На лекции Q-learning по логу случайных блужданий восстановил ценности лучшей стратегии, а SARSA — ценности той стратегии, что собирала лог. Там мир был маленький и ответ сверялся с точной таблицей. Здесь ответ проще и жёстче: **довезёт ли такси**.

Готовая `collect_random_log` даёт 3000 эпизодов случайного таксиста: каждый переход записан пятёркой $(s, a, r, s', a')$ плюс флаг конца эпизода. Напишите `learn_from_log(log, step_fn, n_states, n_actions, alpha, gamma)` — один проход по логу, на каждом переходе одно обновление `step_fn`; обучите им две таблицы, с `q_step` и с `sarsa_step`, и оцените обе жадной стратегией через `average_greedy_return`. Больше ничего менять нельзя: данные одни, цикл один, различаются только правила обновления.

Перед запуском напечатайте, сколько в логе переходов и сколько из них — успешные высадки (награда $+20$). Нарисуйте жадный эпизод такси по таблице Q-learning: `play_episode` даёт состояния, `draw_taxi` рисует первое и последнее (в ряд).

Ответьте текстом: почему по одному и тому же логу Q-learning собрал рабочего таксиста, а SARSA — нет; что именно выучила SARSA-таблица (чья это ценность) и почему её жадная стратегия так плоха.

In [ ]:
taxi = make_taxi()
log = collect_random_log(taxi, n_episodes=3000, seed=0)
print(f"в логе {len(log)} переходов, успешных высадок {sum(1 for step in log if step[2] == 20)}")

def learn_from_log(log, step_fn, n_states, n_actions, alpha=0.1, gamma=1.0):
    # TODO 2.1: таблица из нулей; один проход по логу — для каждой пятёрки (s, a, r, s2, a2, done) один step_fn
    raise NotImplementedError

# TODO 2.1: Q_log_ql и Q_log_sarsa по одному и тому же log; напечатать жадный return обеих (average_greedy_return)
Q_log_ql, Q_log_sarsa = None, None
# TODO 2.1: жадный эпизод по Q_log_ql — первое и последнее состояния через draw_taxi в ряд

In [ ]:
# Самопроверка 2.1
assert Q_log_ql is not None and Q_log_ql.shape == (500, 6), "таблица такси — 500 × 6"
_ql, _sarsa = average_greedy_return(taxi, Q_log_ql), average_greedy_return(taxi, Q_log_sarsa)
assert _ql > 5, f"Q-learning по логу случайной стратегии должен довозить (эталон около +8), у вас {_ql:+.1f}"
assert _sarsa < 0, f"SARSA по тому же логу довозить не должен (эталон около −130), у вас {_sarsa:+.1f}"
print(f"OK: один лог, два правила — {_ql:+.1f} против {_sarsa:+.1f}")

_Ваш ответ:_ TODO

## Часть 3. Expected SARSA на такси (20 баллов)

*Это раздел 7.1 лекции 4 на такси из семинара 4.* На семинаре SARSA с постоянным $\varepsilon = 0.1$ давал на такси плохую жадную стратегию, и лечением было затухание $\varepsilon$. Есть второе лекарство — убрать из цели жребий:

$$
\text{цель} = r + \gamma \sum_{a'} \pi(a' \mid s')\, Q(s', a'),
\qquad
\pi(a' \mid s') = \frac{\varepsilon}{|A|} + (1 - \varepsilon)\,[a' = \arg\max Q(s', \cdot)].
$$

### 3.1 Правило обновления (8 баллов)

Напишите `make_expected_sarsa_step(eps)`: она возвращает функцию с той же сигнатурой, что `sarsa_step` и `q_step`, — тогда её можно отдать в ваш `train`. Вероятности $\pi(\cdot \mid s')$ считайте по формуле выше: $\varepsilon / |A|$ каждому действию и $1 - \varepsilon$ сверху лучшему.

In [ ]:
def make_expected_sarsa_step(eps):
    def expected_sarsa_step(Q, s, a, r, s2, a2, done, alpha, gamma):
        # TODO 3.1: вероятности ε-жадной стратегии в строке s2, цель r + gamma * (probs @ Q[s2]), сдвиг Q[s, a]
        raise NotImplementedError
    return expected_sarsa_step

In [ ]:
# Самопроверка 3.1 — та же ситуация, что в 1.1: строка s2 = [0, 0, 0.50, 0.90], eps = 0.2
Q = np.zeros((2, 4)); Q[0, 0] = 0.30; Q[1, 2] = 0.50; Q[1, 3] = 0.90
make_expected_sarsa_step(0.2)(Q, s=0, a=0, r=0.0, s2=1, a2=2, done=False, alpha=0.1, gamma=1.0)
assert np.isclose(Q[0, 0], 0.349), ("среднее по стратегии: 0.85 · 0.90 + 0.05 · 0.50 = 0.79, "
                                     f"сдвиг 0.30 + 0.1 · (0.79 − 0.30) = 0.349, у вас {Q[0, 0]:.3f}")
make_expected_sarsa_step(0.2)(Q, s=0, a=1, r=1.0, s2=1, a2=2, done=True, alpha=0.5, gamma=1.0)
assert np.isclose(Q[0, 1], 0.5), "после терминального перехода цель — просто r"
print("OK: Expected SARSA — между SARSA (0.32) и Q-learning (0.36) в той же ситуации: 0.349")

### 3.2 Сравнение на такси (12 баллов)

Обучите на такси три алгоритма одной и той же функцией `train` при **постоянном** $\varepsilon = 0.1$, $\alpha = 0.3$, $\gamma = 1$, 5000 эпизодов по 200 шагов: `q_step`, `sarsa_step` и `make_expected_sarsa_step(0.1)`. Для сида 0 нарисуйте три кривые обучения на одном графике (`plot_learning_curves`). Затем для SARSA и Expected SARSA повторите обучение с сидами 0, 1, 2 и напечатайте таблицу: сид, жадный return SARSA, жадный return Expected SARSA.

Ответьте текстом: (1) кривые обучения трёх алгоритмов почти совпадают, а жадные результаты SARSA и Expected SARSA различаются сильно — почему по кривой обучения этого не видно; (2) что в цели SARSA делает жадную стратегию неустойчивой на такси и почему среднее по стратегии это убирает; (3) чем это лекарство отличается от затухания $\varepsilon$ из семинара — что общего у двух способов и когда предпочесть какой.

In [ ]:
# TODO 3.2: три обучения с сидом 0 (q_step, sarsa_step, make_expected_sarsa_step(0.1)) — кривые через plot_learning_curves
# TODO 3.2: сиды 0, 1, 2 для sarsa_step и Expected SARSA — таблица жадных return'ов; результаты Expected SARSA
#           сохранить в список expected_results
expected_results = None

In [ ]:
# Самопроверка 3.2
assert expected_results is not None and len(expected_results) == 3, "expected_results — три жадных return'а, по сиду"
assert min(expected_results) > 5, f"Expected SARSA при постоянном ε должен довозить на всех трёх сидах, у вас {expected_results}"
print(f"OK: Expected SARSA — {[f'{x:+.1f}' for x in expected_results]}")

_Ваш ответ:_ TODO

## Часть 4. Горная машина на сетке (20 баллов)

*Новая среда — и первый шаг к блоку 3.* В `MountainCar-v0` машина стоит в долине между двумя горами; мотор слабый, в лоб на правую гору не въехать, нужно раскачаться. Наблюдение — два числа: позиция $x \in [-1.2, 0.6]$ и скорость $v \in [-0.07, 0.07]$; действия — толкать влево, не толкать, толкать вправо; награда $-1$ за каждый шаг; эпизод кончается у флага ($x \ge 0.5$) или через 200 шагов.

Состояние непрерывное, таблица Q по нему не строится. Старый приём — **разрезать на сетку**: $x$ и $v$ делим на `bins` одинаковых отрезков, состояние — пара номеров отрезков, таблица $Q$ размера `bins × bins × 3`.

### 4.1 Сетка (6 баллов)

Напишите `discretize(obs, bins)`: по `low` и `high` среды переведите наблюдение в пару целых от 0 до `bins − 1` (на верхней границе — `bins − 1`, а не `bins`).

In [ ]:
car = make_mountain_car()
LOW, HIGH = car.observation_space.low, car.observation_space.high

def discretize(obs, bins):
    # TODO 4.1: доля пути от LOW до HIGH, умноженная на bins, с обрезкой до bins − 1; вернуть кортеж из двух int
    raise NotImplementedError

In [ ]:
# Самопроверка 4.1
assert discretize(LOW, 20) == (0, 0) and discretize(HIGH, 20) == (19, 19), "границы попадают в крайние клетки"
assert discretize([-0.5, 0.0], 20) == (7, 10), "дно долины x = −0.5 при нулевой скорости — клетка (7, 10)"
assert isinstance(discretize(LOW, 6)[0], int), "номера клеток — целые числа"
print("OK: сетка")

### 4.2 Q-learning на сетке (14 баллов)

Каркас `train_on_grid` ниже повторяет ваш `train` для состояния-кортежа: таблица `Q[s]` — строка из трёх чисел, где `s = discretize(obs, bins)`. Допишите обновление Q-learning (одна строка) и обучите три сетки: `bins = 6, 20, 40`, по 5000 эпизодов, $\alpha = 0.1$, $\varepsilon = 0.1$, $\gamma = 0.99$. Это самая долгая ячейка ноутбука — около минуты.

Для каждой сетки напечатайте долю эпизодов с финишем за последние 500 эпизодов обучения и среднюю длину **жадного** эпизода по 20 стартам (`greedy_episode`; эпизод без финиша считается за 200 шагов). Нарисуйте две картинки: тепловую карту $V = \max_a Q$ для сетки 20 (`plot_value_grid`) и позицию машины по шагам для жадных эпизодов трёх сеток на одном графике (`plot_positions`).

Ответьте текстом: (1) почему сетка 6 не доезжает, а 20 доезжает — что теряется при слишком грубом разрезании; (2) почему сетка 40 за те же 5000 эпизодов выучивается хуже, чем 20, хотя она точнее; (3) что происходит с таблицей, если состояние — не два числа, а картинка с экрана, и чем это лечат в блоке 3.

In [ ]:
def train_on_grid(env, bins, n_episodes=5000, alpha=0.1, eps=0.1, gamma=0.99, seed=0):
    """Q-learning на сетке bins × bins; возвращает Q и список флагов «доехала» по эпизодам."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((bins, bins, env.action_space.n))
    reached = []
    for episode in range(n_episodes):
        obs, _ = env.reset(seed=int(rng.integers(1_000_000)))
        s = discretize(obs, bins)
        while True:
            a = int(rng.integers(env.action_space.n)) if rng.random() < eps else int(np.argmax(Q[s]))
            obs, r, terminated, truncated, _ = env.step(a)
            s2 = discretize(obs, bins)
            # TODO 4.2: одна строка — сдвиг Q[s][a] к цели r + gamma * max Q[s2] (при terminated будущего нет)
            raise NotImplementedError
            s = s2
            if terminated or truncated:
                reached.append(terminated)
                break
    return Q, np.array(reached)

# TODO 4.2: три сетки (6, 20, 40): обучить, напечатать долю финишей за последние 500 эпизодов и среднюю длину
#           жадного эпизода по 20 стартам; таблицы сохранить в словарь grids = {bins: Q}
grids = None
# TODO 4.2: тепловая карта для сетки 20 и позиции по шагам жадных эпизодов трёх сеток

In [ ]:
# Самопроверка 4.2
assert grids is not None and set(grids) == {6, 20, 40} and grids[20].shape == (20, 20, 3), "grids — таблицы трёх сеток"
_len = {bins: np.mean([len(greedy_episode(car, grids[bins], lambda o, b=bins: discretize(o, b), seed=1000 + k)[0]) - 1
                       for k in range(20)]) for bins in grids}
assert _len[20] < 190, f"сетка 20 должна доезжать: средняя длина жадного эпизода < 190, у вас {_len[20]:.0f}"
assert _len[6] > _len[20], f"сетка 6 хуже сетки 20: {_len[6]:.0f} против {_len[20]:.0f} шагов"
print(f"OK: средняя длина жадного эпизода — сетка 6: {_len[6]:.0f}, 20: {_len[20]:.0f}, 40: {_len[40]:.0f}")

_Ваш ответ:_ TODO

## Часть 5. Теория (10 баллов)

Отвечайте прямо в markdown-ячейке ниже, используя LaTeX. Показывайте промежуточные шаги, а не только ответ.

1. **Off-policy по формуле.** Выпишите цели обновления Q-learning и SARSA. Покажите, что в цели Q-learning стратегия поведения не участвует вовсе, а в цели SARSA участвует через $a'$, и что поэтому неподвижная точка Q-learning — уравнение оптимальности для $Q^*$, а неподвижная точка SARSA — уравнение для $Q^\pi$ той стратегии, которая выбирала $a'$. Объясните этим результат части 2.
2. **Максимум завышает.** Пусть $X_1, \ldots, X_n$ — независимые случайные оценки с $\mathbb E X_i = 0$. Докажите, что $\mathbb E[\max_i X_i] \ge \max_i \mathbb E X_i = 0$, и объясните, почему при $n \ge 2$ и невырожденных $X_i$ неравенство строгое. Для двух автоматов с выигрышем $\pm 1$ с равными шансами посчитайте $\mathbb E[\max(X_1, X_2)]$ по оценкам одной игры на каждом.
3. **Цель TD против цели Монте-Карло.** Запишите обе цели для $V^\pi(s)$. Какая из них несмещённая при любой таблице $V$, а какая — только когда $V = V^\pi$? У какой дисперсия больше и за счёт чего? Одним предложением: почему тем не менее TD часто учится быстрее.

_Ваш ответ:_ TODO

## Чек-лист перед сдачей

- [ ] 1.1: `epsilon_greedy`, `sarsa_step`, `q_step`, `train` перенесены, самопроверка пройдена
- [ ] 1.2: `value_iteration` даёт $V^*(\text{старт}) = 0.95^{13}$, есть печать по четырём бюджетам эпизодов, три картинки и две ошибки, ответ
- [ ] 2.1: `learn_from_log` обучил две таблицы по одному логу, напечатаны размер лога и число высадок, картинка старта и финиша такси, ответ
- [ ] 3.1: `make_expected_sarsa_step` проходит самопроверку
- [ ] 3.2: график трёх кривых, таблица по трём сидам, ответ
- [ ] 4.1: `discretize` проходит самопроверку
- [ ] 4.2: напечатаны доли финишей и длины жадных эпизодов для трёх сеток, тепловая карта и позиции по шагам, ответ
- [ ] 5: три теоретических ответа
- [ ] Коды квизов по лекциям 3 и 4 лежат в `quiz/week03.txt` и `quiz/week04.txt`
- [ ] Ноутбук выполнен целиком: «Restart & Run All» перед сдачей